# Pretrained Transformer: RoBERTa Sentiment Analysis

Use a pretrained HuggingFace transformer (`cardiffnlp/twitter-roberta-base-sentiment-latest`) for zero-shot sentiment classification. Unlike the previous models, this requires **no training** — the model was already fine-tuned on ~124M tweets for 3-class sentiment (Negative / Neutral / Positive).

This serves as a comparison point: how does a pretrained transformer perform on employee reviews without any task-specific training?

## Imports

In [ ]:
import subprocess
subprocess.check_call(['pip', 'install', 'transformers', '-q'])

import warnings
warnings.filterwarnings('ignore')

import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, auc, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from transformers import pipeline
import joblib
import os
from tqdm import tqdm

sns.set_style('whitegrid')
print('All imports successful')

## Helper Classes

In [ ]:
class PretrainedTransformerModel:
    """Pretrained RoBERTa sentiment model from HuggingFace."""

    def __init__(self, str_bucket, str_dirname_output, str_model_name):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.str_model_name = str_model_name
        self.df_test = None
        self.arr_y_test = None
        self.arr_y_pred = None
        self.arr_y_proba = None
        self.cls_pipeline = None
        self.dict_label_map = {
            'negative': 0, 'Negative': 0, 'LABEL_0': 0,
            'neutral': 1, 'Neutral': 1, 'LABEL_1': 1,
            'positive': 2, 'Positive': 2, 'LABEL_2': 2
        }
        self.dict_class_labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

    def load_model(self):
        """Load the pretrained sentiment pipeline from HuggingFace."""
        print(f'Loading pretrained model: {self.str_model_name}')
        self.cls_pipeline = pipeline(
            'sentiment-analysis',
            model=self.str_model_name,
            tokenizer=self.str_model_name,
            top_k=3,
            truncation=True,
            max_length=512
        )
        print(f'Model loaded successfully')

    def import_data(self):
        """Load test split from S3."""
        str_uri = f's3://{self.str_bucket}/02_preprocessing/test_split.parquet'
        self.df_test = pd.read_parquet(str_uri)
        self.arr_y_test = self.df_test['sentiment_3class'].values
        print(f'Loaded test data: {len(self.df_test)} samples')

    def predict(self, int_batch_size=32):
        """Run inference on the test set."""
        list_texts = self.df_test['review_text_clean'].tolist()
        int_n = len(list_texts)

        arr_y_pred = np.zeros(int_n, dtype=int)
        arr_y_proba = np.zeros((int_n, 3), dtype=float)

        print(f'Running inference on {int_n} samples (batch_size={int_batch_size})...')
        for int_start in tqdm(range(0, int_n, int_batch_size)):
            int_end = min(int_start + int_batch_size, int_n)
            list_batch = list_texts[int_start:int_end]

            # Handle empty strings
            list_batch_clean = [str(t) if t and str(t).strip() else 'neutral' for t in list_batch]

            list_results = self.cls_pipeline(list_batch_clean)

            for int_i, list_scores in enumerate(list_results):
                int_idx = int_start + int_i
                # list_scores is a list of dicts: [{'label': 'positive', 'score': 0.9}, ...]
                for dict_score in list_scores:
                    str_label = dict_score['label']
                    int_class = self.dict_label_map.get(str_label, 1)
                    arr_y_proba[int_idx, int_class] = dict_score['score']

                arr_y_pred[int_idx] = np.argmax(arr_y_proba[int_idx])

        self.arr_y_pred = arr_y_pred
        self.arr_y_proba = arr_y_proba
        print(f'Inference complete')

    def evaluate(self):
        """Evaluate predictions against ground truth."""
        flt_accuracy = accuracy_score(self.arr_y_test, self.arr_y_pred)
        flt_f1_macro = f1_score(self.arr_y_test, self.arr_y_pred, average='macro')
        flt_f1_weighted = f1_score(self.arr_y_test, self.arr_y_pred, average='weighted')

        arr_y_bin = label_binarize(self.arr_y_test, classes=[0, 1, 2])
        flt_roc_auc = roc_auc_score(arr_y_bin, self.arr_y_proba, multi_class='ovr', average='macro')

        print('\n' + '='*60)
        print('TEST SET EVALUATION - PRETRAINED ROBERTA')
        print('='*60)
        print(f'Accuracy:      {flt_accuracy:.4f}')
        print(f'F1-Macro:      {flt_f1_macro:.4f}')
        print(f'F1-Weighted:   {flt_f1_weighted:.4f}')
        print(f'ROC AUC:       {flt_roc_auc:.4f}')
        print(f'\nClassification Report:')
        print(classification_report(
            self.arr_y_test, self.arr_y_pred,
            target_names=list(self.dict_class_labels.values()),
            zero_division=0
        ))
        print('='*60)

    def plot_confusion_matrix(self):
        """Plot confusion matrix."""
        arr_cm = confusion_matrix(self.arr_y_test, self.arr_y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(arr_cm, annot=True, fmt='d', cmap='Purples',
                    xticklabels=list(self.dict_class_labels.values()),
                    yticklabels=list(self.dict_class_labels.values()))
        plt.title('Confusion Matrix - Pretrained RoBERTa', fontsize=14, fontweight='bold')
        plt.ylabel('True Label', fontsize=12)
        plt.xlabel('Predicted Label', fontsize=12)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_confusion_matrix.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 01_confusion_matrix.png')

    def plot_roc_curves(self):
        """Plot ROC curves."""
        arr_y_bin = label_binarize(self.arr_y_test, classes=[0, 1, 2])
        list_colors = ['#d62728', '#ff7f0e', '#2ca02c']
        list_labels = ['Negative', 'Neutral', 'Positive']

        fig, ax = plt.subplots(figsize=(10, 8))
        for int_i in range(3):
            fpr, tpr, _ = roc_curve(arr_y_bin[:, int_i], self.arr_y_proba[:, int_i])
            flt_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=list_colors[int_i], lw=2,
                    label=f'{list_labels[int_i]} (AUC = {flt_auc:.3f})')

        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title('ROC Curves - Pretrained RoBERTa', fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=10)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_roc_curves.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 02_roc_curves.png')

    def save_predictions(self):
        """Save predictions for the comparison notebook."""
        dict_preds = {
            'y_pred': self.arr_y_pred,
            'y_proba': self.arr_y_proba
        }
        joblib.dump(dict_preds, f'{self.str_dirname_output}/roberta_predictions.pkl')
        print(f'Predictions saved: {self.str_dirname_output}/roberta_predictions.pkl')

## Constants

In [ ]:
str_bucket = 'nlp-sentiment-analysis-demo'
str_dirname_output = './output'
str_model_name = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
int_batch_size = 32

print('Constants defined')

## Output Directory

In [ ]:
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Output directory ready: {str_dirname_output}')

## Load Model and Data

In [ ]:
cls_model = PretrainedTransformerModel(str_bucket, str_dirname_output, str_model_name)
cls_model.load_model()
cls_model.import_data()

## Run Inference

In [ ]:
cls_model.predict(int_batch_size=int_batch_size)

## Evaluate

In [ ]:
cls_model.evaluate()

In [ ]:
cls_model.plot_confusion_matrix()

In [ ]:
cls_model.plot_roc_curves()

## Save Predictions

In [ ]:
cls_model.save_predictions()

## Summary

In [ ]:
print('\n' + '='*60)
print('PRETRAINED ROBERTA SENTIMENT ANALYSIS COMPLETE')
print('='*60)
print(f'Model: {str_model_name}')
print(f'Test samples: {len(cls_model.df_test)}')
print(f'No training required - pretrained on ~124M tweets')
print(f'Output: {str_dirname_output}')
print('='*60)